# territory

> Squads with soft territories (speedup plan Part B): home ranges as arcs of the location ring, `TerritoryBandit` (a `CommBandit` with a home bias `beta`, from free movement at 0 to hard borders at `inf`), and the share of stays at home.

In [ ]:
#| default_exp territory

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from collections import deque

import numpy as np

from explRL.harness_agents import CommBandit


def homeRanges(L, k):
    "`k` contiguous arcs of `L // k` locations covering 0 .. L - 1 (the last arc takes the remainder)."
    size = L // k
    return [frozenset(range(s * size, L if s == k - 1 else (s + 1) * size)) for s in range(k)]


def squadHomes(L, N, k):
    "Home range of each of N agents in k squads: agent i is in squad i // (N // k)."
    arcs = homeRanges(L, k)
    return [arcs[min(i // max(N // k, 1), k - 1)] for i in range(N)]


class TerritoryBandit(CommBandit):
    """A `CommBandit` with a home range: `beta` is added to the choice score of home locations (soft territory);
    unknown home locations are tried before unknown others; `beta = inf` never leaves home; `beta = 0` is exactly
    `CommBandit`."""
    def __init__(self, channel, home, beta, **kw):
        super().__init__(channel, **kw)
        self.home, self.beta = frozenset(home), float(beta)

    def act(self, obs):
        if self.beta == 0:
            return super().act(obs)
        T = obs['travel']
        if self.rates is None:
            self.rates = [deque(maxlen=self.K) for _ in T]
        mean, n = self._stats(obs)
        home = np.zeros(len(T), bool)
        home[sorted(self.home)] = True
        allowed = home if np.isinf(self.beta) else np.ones(len(T), bool)
        unknown = (n == 0) & allowed
        if unknown.any():
            first = unknown & home if (unknown & home).any() else unknown
            idx = np.flatnonzero(first)
            return int(idx[np.argmin(T[idx])])
        score = (mean + self.cUCB * np.sqrt(np.log1p(n.sum()) / np.where(n > 0, n, 1))) * self.H / (self.H + T)
        bonus = np.where(home, 0.0 if np.isinf(self.beta) else self.beta, 0.0)
        return int(np.argmax(np.where(allowed, score + bonus, -np.inf)))


def homeShare(L, homes):
    "Mean over agents with a home of the share of their stays (ledger rows of kind 'stay') inside it; nan if none."
    shares = []
    for i, h in enumerate(homes):
        if h is None:
            continue
        m = (L['agent'] == i) & (L['kind'] == 'stay')
        if m.any():
            shares.append(float(np.isin(L['location'][m], sorted(h)).mean()))
    return float(np.mean(shares)) if shares else float('nan')

## Tests

In [ ]:
import json
import numpy as np
from explRL.territory import *
from explRL.harness import Blackboard
from explRL.harness_agents import CommBandit
from explRL.harness_runs import benchWorld
from explRL.multi import runTeam
from explRL.team_refs import ledgerArrays

arcs = homeRanges(64, 4)
assert [len(a) for a in arcs] == [16] * 4 and set().union(*arcs) == set(range(64)) and sum(map(len, arcs)) == 64
assert [len(a) for a in homeRanges(64, 5)] == [12, 12, 12, 12, 16]
assert squadHomes(64, 8, 4)[0] == squadHomes(64, 8, 4)[1] == arcs[0] and squadHomes(64, 8, 4)[7] == arcs[3]

def team(cls, beta=None, N=4, seed=5, T=300.0):
    w = benchWorld(tStay=5.0); w.reset(seed, N); bb = Blackboard(64, None, 0.01, N)
    homes = squadHomes(64, N, N // 2)
    ms = [CommBandit(bb) if cls is CommBandit else TerritoryBandit(bb, homes[i], beta) for i in range(N)]
    runTeam(ms, w, T)
    return w, homes

# beta = 0 makes exactly CommBandit's choices
w0, _ = team(CommBandit); w1, _ = team(TerritoryBandit, 0.0)
assert json.dumps(w0.ledger, default=str) == json.dumps(w1.ledger, default=str)
# beta = inf never leaves home after the first move; a larger beta keeps agents home more
w, homes = team(TerritoryBandit, np.inf)
L = ledgerArrays(w.ledger)
assert homeShare(L, homes) == 1.0
shares = [homeShare(ledgerArrays(team(TerritoryBandit, b)[0].ledger), homes) for b in (0.0, 1.0)]
assert shares[1] > shares[0]
assert np.isnan(homeShare(L, [None] * 4))

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()